# Step 1: fact preparation (template choice, candidate facts, keep facts answered correctly in EN, ES and HI) for Llama-3.2-3B-Instruct and Gemma-2-2B
Inputs: the babel-bundle dataset only. Output: `babel_new_results.zip` (also `results/`), which the next step takes as input.

In [ ]:
import os, subprocess, sys, zipfile
STEP = "s01_facts"
HOURS = 10.5  # jobs stop cleanly after this, then everything is packaged
INPUT = os.environ.get("BABEL_INPUT", "/kaggle/input")


def find_driver():
    # any folder under the input that holds the runner files, at any depth (Kaggle mounts datasets at varying paths)
    for root, dirs, files in os.walk(INPUT, followlinks=True):
        if {"driver.py", "steps.py", "dtype_check.py"} <= set(files):
            return os.path.join(root, "driver.py")
    for root, dirs, files in os.walk(INPUT, followlinks=True):  # the zip itself, not unpacked
        if "babel_bundle.zip" in files:
            zipfile.ZipFile(os.path.join(root, "babel_bundle.zip")).extractall("/tmp/babel_src")
            return "/tmp/babel_src/kaggle/driver.py"
    return None


def show_input(depth=4):
    for root, dirs, files in os.walk(INPUT, followlinks=True):
        d = root[len(INPUT):].count(os.sep)
        if d <= depth:
            print("  " * d + (os.path.basename(root) or INPUT) + "/  (" + str(len(files)) + " files, " + str(len(dirs)) + " folders)" + ("  e.g. " + ", ".join(sorted(files)[:4]) if files else ""))
        if d >= depth:
            dirs[:] = []


drv = find_driver()
if not drv:
    print("No kaggle/driver.py anywhere under " + INPUT + ". The attached dataset does not hold the new bundle (an older version is attached, or the new files were not added). What is attached:")
    show_input()
else:
    print("runner found:", drv)
    env = dict(os.environ, BABEL_STEP=STEP, HOURS=str(HOURS), PYTHONUNBUFFERED="1")
    try:
        from kaggle_secrets import UserSecretsClient
        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN secret not readable here (" + type(e).__name__ + "); the driver will try again itself.")
    p = subprocess.Popen([sys.executable, drv], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("driver exit code:", p.returncode, "(0 = everything finished)")

In [ ]:
import json, os
f = "/kaggle/working/results/RUN_SUMMARY.json"
print(json.dumps(json.load(open(f)), indent=1) if os.path.exists(f) else "no summary (see the messages above)")